# Step 5 — Lagrange Multiplier Estimation (Calculations L1–L3)

**Purpose:** Estimate the Lagrange multipliers λ_S and λ_R that characterise the trade-off between the three maturation objectives.

- **L1** — Global λ: fit the KKT stationarity condition across all memory sequences.
- **L2** — Per-germline λ: stratified by IGHV gene (≥1 000 sequences).
- **L3** — Per-donor λ: stratified by donor (≥5 000 memory sequences).

## Theoretical basis

The maturation process minimises the affinity deficit Φ_A subject to:
- structural integrity: Φ_S ≤ c_S
- autoreactivity: Φ_R ≤ c_R

The Lagrangian is `L = Φ_A + λ_S·Φ_S + λ_R·Φ_R`. The KKT stationarity condition at the optimum:

```
∇Φ_A = −λ_S ∇Φ_S − λ_R ∇Φ_R
```

Integrated over a maturation trajectory from the germline baseline:

```
ΔΦ_A(x) ≈ −λ_S · ΔΦ_S(x) − λ_R · ΔΦ_R(x)   (KKT cross-sectional form)
```

where **ΔΦ_k(x) = Φ_k(x) − ⟨Φ_k⟩_germline(x)** is the deviation from the germline average.  
λ_S and λ_R are non-negative (dual feasibility).

## Per-sequence Φ_S construction

Φ_S(x) is the accumulated structural penalty of the mutations in sequence x:

```
Φ_S(x) ≈ n_mut_CDR(x) · ⟨φ_S⟩_CDR  +  n_mut_FWR(x) · ⟨φ_S⟩_FWR
```

where `⟨φ_S⟩_region = mean_i [ max(0, −log ω_i) ]` for IMGT positions i in that region,  
computed from `omega_per_position.parquet` (Step 2 output).  
CDR = CDR1 + CDR2 in the VH V-region (not CDR3; CDR3 contributes via Φ_R).  
FWR = FR1 + FR2 + FR3.  
Since CDR positions are under positive selection (ω ≥ 1), ⟨φ_S⟩_CDR ≈ 0; the structural  
penalty is dominated by FWR mutations at constrained positions.

## Adaptations from Steps 2–4

- Φ_S^interface = 0 (Step 2: no VH–VL coupling detected) — omitted from the model.
- Φ_R decomposes into CDR3 feature component (captured by logistic model, Step 4) and germline  
  framework component (captured by per-germline demeaning in L1 and explicitly in L2).
- Isotype subclass collapse applied in Steps 3–4; inherited here.
- Q_H3 anomaly (positive coefficient in logistic model) does not affect the regression directly;  
  Φ_R is used as a scalar already baked from the model output.

**Inputs:** `results/tables/omega_per_position.parquet`, `results/tables/affinity_proxy.parquet`,  
`results/tables/phi_r_scores.parquet`  
**Outputs:** `results/tables/lambda_global.csv`, `results/tables/lambda_by_germline.csv`,  
`results/tables/lambda_by_donor.csv`, `results/figures/fig_l1_*.png`, `results/figures/fig_l2_*.png`,  
`results/figures/fig_l3_*.png`

In [ ]:
import polars as pl
import numpy as np
import math
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from scipy.optimize import nnls, minimize
from scipy.stats import pearsonr, spearmanr

In [ ]:
DATA_DIR = Path("/home/jovyan/shared/Benjamin/LineageAtlas/pairplex_paper/")
RESULTS  = DATA_DIR / "results"
FIGURES  = RESULTS / "figures"
TABLES   = RESULTS / "tables"

print("Paths OK")

In [ ]:
# ── Load omega per position, compute mean structural cost per IMGT region ─────
# phi_S at position i = max(0, -ln ω_i): penalty for mutations at constrained sites
# CDR positions (ω ≥ 1 typically) → clipped to 0  (no structural penalty)
# FWR positions (ω << 1)           → -ln(ω) > 0    (strong structural penalty)

omega_df = pl.read_parquet(TABLES / "omega_per_position.parquet")

print(f"omega_per_position: {omega_df.shape}")
print(f"Regions present: {sorted(omega_df['region'].unique().to_list())}")
print(f"Omega range (finite): "
      f"{omega_df.filter(pl.col('omega').is_not_null())['omega'].min():.4f} – "
      f"{omega_df.filter(pl.col('omega').is_not_null())['omega'].max():.4f}")

# Compute phi_S_local per position
omega_df = omega_df.with_columns(
    pl.when(pl.col('omega').is_not_null() & (pl.col('omega') > 0))
    .then((-pl.col('omega').log(math.e)).clip(lower_bound=0.0))
    .otherwise(0.0)
    .alias('phi_s_local')
)

# Regional mean structural cost
region_phi_s = (
    omega_df
    .group_by('region')
    .agg([
        pl.col('phi_s_local').mean().alias('mean_phi_s'),
        pl.col('phi_s_local').median().alias('median_phi_s'),
        pl.col('omega').mean().alias('mean_omega'),
        pl.len().alias('n_positions'),
    ])
    .sort('region')
)
print("\nMean structural cost per region:")
print(region_phi_s)

# Extract regional means as scalars
region_dict = {r['region']: r['mean_phi_s'] for r in region_phi_s.iter_rows(named=True)}

# CDR = CDR1 + CDR2 average
mean_phi_s_CDR = np.mean([
    region_dict.get('CDR1', 0.0),
    region_dict.get('CDR2', 0.0),
])
# FWR = FR1 + FR2 + FR3 average
mean_phi_s_FWR = np.mean([
    region_dict.get('FR1', 0.0),
    region_dict.get('FR2', 0.0),
    region_dict.get('FR3', 0.0),
])

print(f"\nMean phi_S cost:")
print(f"  CDR (CDR1+CDR2 average): {mean_phi_s_CDR:.4f}")
print(f"  FWR (FR1+FR2+FR3 average): {mean_phi_s_FWR:.4f}")
print(f"  → Structural penalty dominated by FWR mutations (as expected)")

In [ ]:
# ── Load affinity proxy and phi_R scores, join on seq_name ───────────────────
print("Loading affinity_proxy...")
phi_a_df = pl.read_parquet(TABLES / "affinity_proxy.parquet")
print(f"  {phi_a_df.shape}  columns: {phi_a_df.columns}")

print("Loading phi_r_scores...")
phi_r_df = pl.read_parquet(TABLES / "phi_r_scores.parquet")
print(f"  {phi_r_df.shape}  columns: {phi_r_df.columns}")

# Inner join: keep only sequences scored for BOTH phi_A and phi_R
data = (
    phi_a_df
    .filter(pl.col('phi_A').is_not_null())
    .select(['seq_name', 'v_gene:0', 'donor', 'lineage', 'isotype_class',
             'n_R_CDR_H', 'n_S_CDR_H', 'n_R_FWR_H', 'n_S_FWR_H',
             'n_mut_H', 'phi_A'])
    .join(
        phi_r_df.select(['seq_name', 'phi_R']),
        on='seq_name', how='inner'
    )
)

print(f"\nJoined dataset: {data.height:,} sequences")
print(f"V-genes present: {data['v_gene:0'].n_unique()}")
print(f"Donors present:  {data['donor'].n_unique()}")

In [ ]:
# ── Compute per-sequence Φ_S from mutation counts × regional mean cost ────────
# n_mut_CDR = n_R_CDR_H + n_S_CDR_H  (CDR1+CDR2 V-region mutations)
# n_mut_FWR = n_R_FWR_H + n_S_FWR_H  (FR1+FR2+FR3 V-region mutations)
# Φ_S(x) ≈ n_mut_CDR × ⟨φ_S⟩_CDR  +  n_mut_FWR × ⟨φ_S⟩_FWR
#
# Biological note: CDR1+CDR2 are under positive selection (ω ≥ 1) for most germlines,
# so ⟨φ_S⟩_CDR ≈ 0. FWR positions are under purifying selection (ω << 1), so
# ⟨φ_S⟩_FWR >> 0. The structural penalty is therefore dominated by FWR mutations.

PHI_S_CDR = float(mean_phi_s_CDR)  # scalar
PHI_S_FWR = float(mean_phi_s_FWR)  # scalar

data = data.with_columns([
    (pl.col('n_R_CDR_H') + pl.col('n_S_CDR_H')).alias('n_mut_CDR_H'),
    (pl.col('n_R_FWR_H') + pl.col('n_S_FWR_H')).alias('n_mut_FWR_H'),
]).with_columns(
    (pl.col('n_mut_CDR_H') * PHI_S_CDR
     + pl.col('n_mut_FWR_H') * PHI_S_FWR).alias('phi_S')
)

print("Per-sequence phi_S statistics:")
print(data['phi_S'].describe())

print("\nPhi_A statistics:")
print(data['phi_A'].describe())

print("\nPhi_R statistics:")
print(data['phi_R'].describe())

# Pairwise Spearman correlations
sample = data.sample(n=100_000, seed=42)
for a, b in [('phi_A','phi_S'), ('phi_A','phi_R'), ('phi_S','phi_R')]:
    r, p = spearmanr(sample[a].to_numpy(), sample[b].to_numpy())
    print(f"  Spearman {a} ~ {b}: r={r:.4f}, p={p:.2e}")

## L1 — Global Lagrange Multipliers

**Setup:** Demean all three objectives by germline mean (within-germline estimator), then fit:

```
−ΔΦ_A(x) = λ_S · ΔΦ_S(x) + λ_R · ΔΦ_R(x)  +  ε
```

using non-negative least squares (NNLS, dual feasibility) and Huber regression (robust to outliers).  
95% bootstrap confidence intervals (n_boot = 500).

**Expected results:**
- λ_S > 0: sequences that accumulated structurally costly FWR mutations traded structural
  integrity for affinity gain (they also made more CDR replacements).
- λ_R > 0: sequences with lower reactivity risk (lower Φ_R) tend to have better affinity
  (lower Φ_A), consistent with the reactivity constraint limiting CDR3 optimisation.
- λ_S and λ_R are dimensionless exchange rates (units: Φ_A / Φ_S and Φ_A / Φ_R).

In [ ]:
# ── Within-germline demeaning ─────────────────────────────────────────────────
# Subtracts the germline-specific mean of each objective, removing the confound
# that different germlines have systematically different baseline Phi values.

germ_means = (
    data
    .group_by('v_gene:0')
    .agg([
        pl.col('phi_A').mean().alias('mean_phi_A_g'),
        pl.col('phi_S').mean().alias('mean_phi_S_g'),
        pl.col('phi_R').mean().alias('mean_phi_R_g'),
    ])
)

data_dm = (
    data
    .join(germ_means, on='v_gene:0', how='left')
    .with_columns([
        (pl.col('phi_A') - pl.col('mean_phi_A_g')).alias('d_phi_A'),
        (pl.col('phi_S') - pl.col('mean_phi_S_g')).alias('d_phi_S'),
        (pl.col('phi_R') - pl.col('mean_phi_R_g')).alias('d_phi_R'),
    ])
)

print(f"Demeaned dataset: {data_dm.height:,} sequences")
print(f"  mean(d_phi_A): {data_dm['d_phi_A'].mean():.6f}  (expected ~0)")
print(f"  mean(d_phi_S): {data_dm['d_phi_S'].mean():.6f}  (expected ~0)")
print(f"  mean(d_phi_R): {data_dm['d_phi_R'].mean():.6f}  (expected ~0)")

In [ ]:
# ── L1 global regression (NNLS + Huber) ───────────────────────────────────────
# Model: −ΔΦ_A = λ_S · ΔΦ_S + λ_R · ΔΦ_R
# NNLS: minimises ||X λ − y||²  subject to λ ≥ 0
# Huber: minimises robust loss, same non-negativity constraint via L-BFGS-B bounds

Y  = -data_dm['d_phi_A'].to_numpy()          # −ΔΦ_A (response)
X  = np.column_stack([
    data_dm['d_phi_S'].to_numpy(),            # ΔΦ_S
    data_dm['d_phi_R'].to_numpy(),            # ΔΦ_R
])

# --- NNLS ---
lambda_nnls, residual_nnls = nnls(X, Y)
lambda_S_nnls, lambda_R_nnls = lambda_nnls
print(f"NNLS global:  λ_S = {lambda_S_nnls:.4f}  |  λ_R = {lambda_R_nnls:.4f}")
print(f"  NNLS residual: {residual_nnls:.4f}")

# R² (explained variance)
Y_pred_nnls = X @ lambda_nnls
ss_res = np.sum((Y - Y_pred_nnls)**2)
ss_tot = np.sum((Y - Y.mean())**2)
r2_nnls = 1 - ss_res / ss_tot
print(f"  R² (NNLS): {r2_nnls:.4f}")

# --- Huber regression (robust) ---
DELTA_HUBER = 1.35   # standard choice for 95% efficiency at Gaussian

def huber_loss(params, X, y, delta=DELTA_HUBER):
    pred   = X @ params
    resid  = y - pred
    abs_r  = np.abs(resid)
    loss   = np.where(
        abs_r <= delta,
        0.5 * resid**2,
        delta * (abs_r - 0.5 * delta)
    )
    return loss.sum()

result_hub = minimize(
    huber_loss,
    x0=lambda_nnls,
    args=(X, Y),
    method='L-BFGS-B',
    bounds=[(0.0, None), (0.0, None)],
)
lambda_S_hub, lambda_R_hub = result_hub.x
print(f"\nHuber global: λ_S = {lambda_S_hub:.4f}  |  λ_R = {lambda_R_hub:.4f}")
print(f"  Huber converged: {result_hub.success}")

# --- Bootstrap CI (NNLS) ---
N_BOOT = 500
rng = np.random.default_rng(42)
boot_lambdas = np.zeros((N_BOOT, 2))

print(f"\nBootstrap CIs (n_boot={N_BOOT})...")
for i in range(N_BOOT):
    idx = rng.integers(0, len(Y), size=len(Y))
    lam_b, _ = nnls(X[idx], Y[idx])
    boot_lambdas[i] = lam_b

ci_lo = np.percentile(boot_lambdas, 2.5,  axis=0)
ci_hi = np.percentile(boot_lambdas, 97.5, axis=0)

print(f"  λ_S = {lambda_S_nnls:.4f}  95% CI [{ci_lo[0]:.4f}, {ci_hi[0]:.4f}]")
print(f"  λ_R = {lambda_R_nnls:.4f}  95% CI [{ci_lo[1]:.4f}, {ci_hi[1]:.4f}]")

# Save
global_result = pl.DataFrame({
    'method':   ['NNLS', 'NNLS', 'Huber', 'Huber'],
    'param':    ['lambda_S', 'lambda_R', 'lambda_S', 'lambda_R'],
    'estimate': [lambda_S_nnls, lambda_R_nnls, lambda_S_hub, lambda_R_hub],
    'ci_lo_95': [ci_lo[0], ci_lo[1], None, None],
    'ci_hi_95': [ci_hi[0], ci_hi[1], None, None],
})
global_result.write_csv(TABLES / "lambda_global.csv")
print(f"\nSaved → {TABLES}/lambda_global.csv")

In [ ]:
# ── L1 visualisation ─────────────────────────────────────────────────────────
sample_idx = rng.integers(0, len(Y), size=50_000)
Y_s    = Y[sample_idx]
X_s    = X[sample_idx]
Ypred  = X_s @ lambda_nnls

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Left: predicted vs actual
ax = axes[0]
ax.scatter(Ypred, Y_s, s=1, alpha=0.05, color='#1E88E5')
lim = np.percentile(np.abs(Y_s), 99)
ax.plot([-lim, lim], [-lim, lim], 'r--', lw=1)
ax.set_xlabel('Predicted −ΔΦ_A  (λ_S·ΔΦ_S + λ_R·ΔΦ_R)')
ax.set_ylabel('Observed −ΔΦ_A')
ax.set_title(f'Global KKT fit\n(R²={r2_nnls:.3f})')

# Middle: λ estimate with bootstrap CI
ax2 = axes[1]
params = ['λ_S', 'λ_R']
vals   = [lambda_S_nnls, lambda_R_nnls]
errs_lo = vals - ci_lo
errs_hi = ci_hi - vals
bars = ax2.barh(params, vals, xerr=[errs_lo, errs_hi], color=['#1E88E5', '#E53935'],
                alpha=0.8, capsize=5)
ax2.axvline(0, color='gray', linestyle='--', lw=1)
ax2.set_xlabel('Lagrange multiplier estimate (NNLS)')
ax2.set_title('Global λ estimates\n(95% bootstrap CI)')

# Add Huber comparison
ax2.scatter([lambda_S_hub, lambda_R_hub], params,
            color='orange', s=80, zorder=5, label='Huber')
ax2.legend(fontsize=8)

# Right: bootstrap distributions
ax3 = axes[2]
ax3.hist(boot_lambdas[:, 0], bins=40, color='#1E88E5', alpha=0.6, label=f'λ_S (mean={lambda_S_nnls:.3f})')
ax3.hist(boot_lambdas[:, 1], bins=40, color='#E53935', alpha=0.6, label=f'λ_R (mean={lambda_R_nnls:.3f})')
ax3.set_xlabel('Bootstrap λ estimate')
ax3.set_ylabel('Count')
ax3.set_title('Bootstrap distributions (n=500)\n(NNLS, germline-demeaned)')
ax3.legend(fontsize=9)

plt.tight_layout()
plt.savefig(FIGURES / "fig_l1_lambda_global.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved.")

## L2 — Per-Germline Lagrange Multipliers

Repeat L1 stratified by IGHV gene for all germlines with ≥1 000 memory sequences.  
Within each germline, the demeaning uses the germline-specific mean (same formula as L1  
but restricted to that stratum).

**Key questions:**
- Do IGHV1-69 / IGHV1-2 (bnAb germlines) show distinct λ coordinates?
- Does IGHV4-34 (autoreactivity-rich) have an anomalously high λ_R?
- Is there a positive correlation between λ_S and λ_R (germlines with tight structural
  constraints also face tighter reactivity constraints)?

Bootstrap CIs are not computed per-germline to limit compute; instead a jackknife SE  
is reported for germlines with ≥5 000 sequences.

In [ ]:
# ── L2 per-germline regression ────────────────────────────────────────────────
MIN_GERM_N = 1000
JACKKNIFE_MIN_N = 5000   # only compute jackknife SE for larger strata

germline_results = []

for vgene, subdf in data.partition_by('v_gene:0', as_dict=True).items():
    n = subdf.height
    if n < MIN_GERM_N:
        continue

    # Demean within this germline stratum
    means = subdf.select([pl.col('phi_A').mean(), pl.col('phi_S').mean(),
                          pl.col('phi_R').mean()]).to_numpy()[0]
    phi_A_g = subdf['phi_A'].to_numpy() - means[0]
    phi_S_g = subdf['phi_S'].to_numpy() - means[1]
    phi_R_g = subdf['phi_R'].to_numpy() - means[2]

    Y_g = -phi_A_g
    X_g = np.column_stack([phi_S_g, phi_R_g])

    lam_g, _ = nnls(X_g, Y_g)

    # R²
    Y_pred_g = X_g @ lam_g
    ss_res_g = np.sum((Y_g - Y_pred_g)**2)
    ss_tot_g = np.sum((Y_g - Y_g.mean())**2)
    r2_g = 1 - ss_res_g / ss_tot_g if ss_tot_g > 0 else np.nan

    # Jackknife SE for large strata
    se_S, se_R = np.nan, np.nan
    if n >= JACKKNIFE_MIN_N:
        # Block jackknife: 20 blocks
        n_blocks = 20
        block_size = n // n_blocks
        jk_lambdas = []
        for b in range(n_blocks):
            mask = np.ones(n, dtype=bool)
            mask[b*block_size : (b+1)*block_size] = False
            lam_b, _ = nnls(X_g[mask], Y_g[mask])
            jk_lambdas.append(lam_b)
        jk_lambdas = np.array(jk_lambdas)
        se_S = np.sqrt((n_blocks - 1) / n_blocks * np.sum((jk_lambdas[:,0] - lam_g[0])**2))
        se_R = np.sqrt((n_blocks - 1) / n_blocks * np.sum((jk_lambdas[:,1] - lam_g[1])**2))

    germline_results.append({
        'v_gene': vgene,
        'lambda_S': float(lam_g[0]),
        'lambda_R': float(lam_g[1]),
        'se_S': float(se_S),
        'se_R': float(se_R),
        'r2': float(r2_g),
        'n': n,
    })

germline_df = pl.DataFrame(germline_results).sort('lambda_S', descending=True)
germline_df.write_csv(TABLES / "lambda_by_germline.csv")

print(f"Germlines with ≥{MIN_GERM_N} sequences: {germline_df.height}")
print(f"\nTop 10 by λ_S:")
print(germline_df.head(10))
print(f"\nIGHV4-34:")
print(germline_df.filter(pl.col('v_gene') == 'IGHV4-34'))
print(f"\nIGHV1-69:")
print(germline_df.filter(pl.col('v_gene') == 'IGHV1-69'))
print(f"\nSaved → {TABLES}/lambda_by_germline.csv")

In [ ]:
# ── L2 visualisation ─────────────────────────────────────────────────────────
lS = germline_df['lambda_S'].to_numpy()
lR = germline_df['lambda_R'].to_numpy()
genes_l2 = germline_df['v_gene'].to_list()
n_seqs = germline_df['n'].to_numpy()

# Highlight special germlines
BNAB_GENES   = {'IGHV1-2', 'IGHV1-69', 'IGHV1-69-2'}
AUTOREACTIVE = {'IGHV4-34'}

colors_l2 = []
sizes_l2  = []
for g, n in zip(genes_l2, n_seqs):
    if g in AUTOREACTIVE:
        colors_l2.append('#FF6F00')   # orange
        sizes_l2.append(120)
    elif g in BNAB_GENES:
        colors_l2.append('#4CAF50')   # green
        sizes_l2.append(120)
    else:
        colors_l2.append('#1E88E5')
        sizes_l2.append(30 + n / 5000)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Left: λ_S vs λ_R scatter
ax = axes[0]
ax.scatter(lS, lR, c=colors_l2, s=sizes_l2, alpha=0.8, zorder=3)

# Label key germlines
for g, s, r in zip(genes_l2, lS, lR):
    if g in BNAB_GENES | AUTOREACTIVE or s > np.percentile(lS, 90) or r > np.percentile(lR, 90):
        ax.annotate(g, (s, r), fontsize=6, ha='left',
                    xytext=(3, 3), textcoords='offset points')

ax.axhline(lambda_R_nnls, color='gray', linestyle=':', lw=1, label='global λ_R')
ax.axvline(lambda_S_nnls, color='gray', linestyle=':', lw=1, label='global λ_S')
ax.set_xlabel('λ_S (structural price per unit Φ_S)')
ax.set_ylabel('λ_R (reactivity price per unit Φ_R)')
ax.set_title('Per-germline Lagrange multipliers\n(size ∝ #sequences)')

legend_handles = [
    mpatches.Patch(color='#FF6F00', label='IGHV4-34 (autoreactive)'),
    mpatches.Patch(color='#4CAF50', label='bnAb germlines'),
    mpatches.Patch(color='#1E88E5', label='Other germlines'),
]
ax.legend(handles=legend_handles, fontsize=8)

# Right: ranked bar plot of λ_R
ax2 = axes[1]
gdf_sorted = germline_df.sort('lambda_R', descending=True)
top_n = min(30, gdf_sorted.height)
g_names = gdf_sorted['v_gene'].to_list()[:top_n]
g_lR    = gdf_sorted['lambda_R'].to_numpy()[:top_n]
g_colors = ['#FF6F00' if g in AUTOREACTIVE else
            '#4CAF50' if g in BNAB_GENES else '#E53935'
            for g in g_names]

ax2.barh(range(top_n), g_lR[::-1], color=g_colors[::-1], alpha=0.8)
ax2.set_yticks(range(top_n))
ax2.set_yticklabels(g_names[::-1], fontsize=8)
ax2.axvline(lambda_R_nnls, color='gray', linestyle='--', lw=1, label=f'global λ_R={lambda_R_nnls:.3f}')
ax2.set_xlabel('λ_R (per-germline)')
ax2.set_title(f'Top {top_n} germlines by λ_R\n(higher = stronger reactivity constraint)')
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig(FIGURES / "fig_l2_lambda_by_germline.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved.")

## L3 — Per-Donor Lagrange Multipliers

Repeat L1 stratified by donor for donors with ≥5 000 memory sequences.  
Within each donor, demean by (donor × germline) to control for germline frequency  
variation between individuals.  

**Key questions:**
- Is there inter-individual variation in λ_S and λ_R?
- Are λ_S and λ_R correlated across donors (a donor with a tight structural budget also  
  faces a tighter reactivity budget)?
- Does the distribution of λ_R across donors reflect known inter-individual variation  
  in autoreactivity levels?

In [ ]:
# ── L3 per-donor regression ───────────────────────────────────────────────────
MIN_DONOR_N = 5000

donor_results = []

for donor, subdf in data.partition_by('donor', as_dict=True).items():
    n = subdf.height
    if n < MIN_DONOR_N:
        continue

    # Demean by (donor × v_gene): control for both donor and germline effects
    vg_means_d = (
        subdf
        .group_by('v_gene:0')
        .agg([
            pl.col('phi_A').mean().alias('m_phi_A'),
            pl.col('phi_S').mean().alias('m_phi_S'),
            pl.col('phi_R').mean().alias('m_phi_R'),
        ])
    )
    subdf_dm = (
        subdf
        .join(vg_means_d, on='v_gene:0', how='left')
        .with_columns([
            (pl.col('phi_A') - pl.col('m_phi_A')).alias('d_phi_A'),
            (pl.col('phi_S') - pl.col('m_phi_S')).alias('d_phi_S'),
            (pl.col('phi_R') - pl.col('m_phi_R')).alias('d_phi_R'),
        ])
    )

    Y_d = -subdf_dm['d_phi_A'].to_numpy()
    X_d = np.column_stack([
        subdf_dm['d_phi_S'].to_numpy(),
        subdf_dm['d_phi_R'].to_numpy(),
    ])

    lam_d, _ = nnls(X_d, Y_d)

    Y_pred_d = X_d @ lam_d
    ss_res_d = np.sum((Y_d - Y_pred_d)**2)
    ss_tot_d = np.sum((Y_d - Y_d.mean())**2)
    r2_d = 1 - ss_res_d / ss_tot_d if ss_tot_d > 0 else np.nan

    donor_results.append({
        'donor':    donor,
        'lambda_S': float(lam_d[0]),
        'lambda_R': float(lam_d[1]),
        'r2':       float(r2_d),
        'n':        n,
    })

donor_df = pl.DataFrame(donor_results).sort('donor')
donor_df.write_csv(TABLES / "lambda_by_donor.csv")

print(f"Donors with ≥{MIN_DONOR_N} memory sequences: {donor_df.height}")
print(f"\nλ_S across donors:")
print(f"  mean={donor_df['lambda_S'].mean():.4f}  std={donor_df['lambda_S'].std():.4f}  "
      f"range=[{donor_df['lambda_S'].min():.4f}, {donor_df['lambda_S'].max():.4f}]")
print(f"\nλ_R across donors:")
print(f"  mean={donor_df['lambda_R'].mean():.4f}  std={donor_df['lambda_R'].std():.4f}  "
      f"range=[{donor_df['lambda_R'].min():.4f}, {donor_df['lambda_R'].max():.4f}]")
print(f"\nSaved → {TABLES}/lambda_by_donor.csv")

In [ ]:
# ── L3 visualisation ─────────────────────────────────────────────────────────
lS_d = donor_df['lambda_S'].to_numpy()
lR_d = donor_df['lambda_R'].to_numpy()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Left: box plots of λ_S and λ_R distribution across donors
ax = axes[0]
bp = ax.boxplot([lS_d, lR_d], labels=['λ_S', 'λ_R'],
                patch_artist=True,
                medianprops={'color': 'white', 'linewidth': 2})
for patch, color in zip(bp['boxes'], ['#1E88E5', '#E53935']):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)

# Add global estimates as reference lines
ax.axhline(lambda_S_nnls, color='#1E88E5', linestyle='--', lw=1.5, alpha=0.6,
           label=f'global λ_S={lambda_S_nnls:.3f}')
ax.axhline(lambda_R_nnls, color='#E53935', linestyle='--', lw=1.5, alpha=0.6,
           label=f'global λ_R={lambda_R_nnls:.3f}')
ax.set_ylabel('Lagrange multiplier value')
ax.set_title(f'λ distribution across donors (n={len(lS_d)})')
ax.legend(fontsize=8)

# Middle: λ_S vs λ_R scatter across donors
ax2 = axes[1]
ax2.scatter(lS_d, lR_d, s=40, alpha=0.7, color='#1E88E5')
r_cor, p_cor = pearsonr(lS_d, lR_d)
ax2.set_xlabel('λ_S (structural price)')
ax2.set_ylabel('λ_R (reactivity price)')
ax2.set_title(f'Per-donor λ_S vs λ_R\n(r={r_cor:.3f}, p={p_cor:.3f})')

# Right: λ_R distribution with kernel density
ax3 = axes[2]
ax3.hist(lR_d, bins=20, color='#E53935', alpha=0.7, density=True)
ax3.axvline(lambda_R_nnls, color='black', lw=1.5, linestyle='--',
            label=f'global λ_R={lambda_R_nnls:.3f}')
ax3.axvline(np.median(lR_d), color='orange', lw=1.5, linestyle='-.',
            label=f'median={np.median(lR_d):.3f}')
ax3.set_xlabel('Per-donor λ_R')
ax3.set_ylabel('Density')
ax3.set_title('Inter-individual variation in λ_R\n(reactivity constraint strength)')
ax3.legend(fontsize=8)

plt.tight_layout()
plt.savefig(FIGURES / "fig_l3_lambda_by_donor.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved.")

In [ ]:
# ── Summary comparison: global vs per-germline vs per-donor ──────────────────
print("=" * 60)
print("LAGRANGE MULTIPLIER SUMMARY")
print("=" * 60)

print(f"\nL1 Global (NNLS, germline-demeaned, n={len(Y):,}):")
print(f"   λ_S = {lambda_S_nnls:.4f}  [95% CI: {ci_lo[0]:.4f}, {ci_hi[0]:.4f}]")
print(f"   λ_R = {lambda_R_nnls:.4f}  [95% CI: {ci_lo[1]:.4f}, {ci_hi[1]:.4f}]")
print(f"   R²  = {r2_nnls:.4f}")

print(f"\nL1 Global (Huber, robust):")
print(f"   λ_S = {lambda_S_hub:.4f}")
print(f"   λ_R = {lambda_R_hub:.4f}")

print(f"\nL2 Per-germline (n={germline_df.height} germlines, ≥1 000 sequences each):")
print(f"   λ_S: mean={germline_df['lambda_S'].mean():.4f}  "
      f"std={germline_df['lambda_S'].std():.4f}  "
      f"range=[{germline_df['lambda_S'].min():.4f}, {germline_df['lambda_S'].max():.4f}]")
print(f"   λ_R: mean={germline_df['lambda_R'].mean():.4f}  "
      f"std={germline_df['lambda_R'].std():.4f}  "
      f"range=[{germline_df['lambda_R'].min():.4f}, {germline_df['lambda_R'].max():.4f}]")

print(f"\nL3 Per-donor (n={donor_df.height} donors, ≥5 000 sequences each):")
print(f"   λ_S: mean={donor_df['lambda_S'].mean():.4f}  "
      f"std={donor_df['lambda_S'].std():.4f}")
print(f"   λ_R: mean={donor_df['lambda_R'].mean():.4f}  "
      f"std={donor_df['lambda_R'].std():.4f}")

# Biological interpretation flag
if lambda_R_nnls > 0:
    print("\n✓ λ_R > 0: reactivity constraint is active — confirmed.")
if lambda_S_nnls > 0:
    print("✓ λ_S > 0: structural constraint is active — confirmed.")
if lambda_S_nnls < 1e-4:
    print("⚠ λ_S ≈ 0: structural penalty may not be limiting (Φ_S^interface=0; FWR mutations\n"
          "   may be near-neutral in this parameterisation). Consider position-level Φ_S.")

## Step 5 Summary

| Calculation | Output table | Output figure(s) |
|-------------|-------------|------------------|
| L1: global λ | `lambda_global.csv` | `fig_l1_lambda_global.png` |
| L2: per-germline λ | `lambda_by_germline.csv` | `fig_l2_lambda_by_germline.png` |
| L3: per-donor λ | `lambda_by_donor.csv` | `fig_l3_lambda_by_donor.png` |

**Key formulae:**
```
Φ_S(x)  = n_mut_CDR_H × ⟨−ln ω⟩_CDR  +  n_mut_FWR_H × ⟨−ln ω⟩_FWR    (clipped at 0)
ΔΦ_k(x) = Φ_k(x) − ⟨Φ_k⟩_v_gene(x)                                     (germline demean)
−ΔΦ_A   = λ_S · ΔΦ_S  +  λ_R · ΔΦ_R  +  ε                              (KKT regression)
λ_S, λ_R ≥ 0                                                              (dual feasibility)
```

**Interpretation of λ values:**
- **λ_S**: structural exchange rate. λ_S = 0.1 means that accumulating one unit of structural
  penalty (one additional constrained FWR mutation at a site with ω ≈ 0.37) correlates
  with a 0.1-unit reduction in affinity deficit.
- **λ_R**: reactivity exchange rate. λ_R = 0.1 means that sequences whose CDRH3 features
  place them one unit below the germline mean reactivity risk have 0.1 units less affinity
  deficit — i.e., reduced reactivity risk is associated with improved affinity.

**Approximation limitations:**
1. Φ_S(x) uses region-level mean omega (not position-specific mutation calls). A more
   accurate version would use the full mutation matrix from aligned_master.parquet.
2. The cross-sectional regression estimates the population-average trade-off, not a
   within-lineage trajectory. The Hamiltonian analysis (Step 7) will test whether
   lineage trajectories follow the predicted gradient direction.
3. The model has no interaction term (λ_S and λ_R are assumed independent). A richer
   model could allow for germline-specific correlations between structural and reactivity
   constraints.

**Next step:** `06_pareto.ipynb` — Pareto front mapping: assign (Φ_S, Φ_A, Φ_R) triplets
to all memory sequences and identify the empirical Pareto front.